# Colab-X-Local-Model server
Run every cell top to bottom. The last cell blocks and prints a `Public tunnel URL` line — copy that URL into `ui/index.html`'s `BASE_URL` constant.

No signup or auth token needed — the tunnel uses Cloudflare's free `cloudflared` quick tunnel. The first run downloads the `cloudflared` binary automatically.

To use a different model, change `MODEL_NAME` in the cell below before running it — nothing else needs to change.

In [ ]:
!pip install -q flask==3.0.3 flask-cors==4.0.1 transformers==4.44.2 torch==2.4.1

In [ ]:
# The one place to change the model — pick per experiment.
MODEL_NAME = "gpt2"

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model=MODEL_NAME)

# Sanity check inside the notebook before wiring up Flask.
print(generator("Hello, my name is", max_new_tokens=20, num_return_sequences=1))

In [ ]:
from flask import Flask, jsonify, request
from flask_cors import CORS

app = Flask(__name__)
CORS(app)


@app.post("/generate")
def generate():
    data = request.get_json(silent=True) or {}
    prompt = data.get("prompt", "")
    if not isinstance(prompt, str) or not prompt.strip():
        return jsonify({"error": "prompt is required"}), 400

    try:
        max_new_tokens = min(int(data.get("max_new_tokens", 200) or 200), 512)
    except (TypeError, ValueError):
        max_new_tokens = 200

    try:
        outputs = generator(prompt, max_new_tokens=max_new_tokens, num_return_sequences=1)
    except Exception as exc:
        return jsonify({"error": str(exc)}), 500

    text = outputs[0]["generated_text"]
    if text.startswith(prompt):
        text = text[len(prompt):].lstrip()
    return jsonify({"response": text})

In [ ]:
# This cell blocks — the tunnel URL is printed above the running server log.
import subprocess
import threading
import re


def run_flask():
    app.run(host="0.0.0.0", port=5000)


threading.Thread(target=run_flask, daemon=True).start()

# Install cloudflared if not already present (Colab has no cloudflared preinstalled).
import os

if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
        "-O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
        shell=True, check=True,
    )

process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:5000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

for line in process.stdout:
    print(line, end="")
    match = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
    if match:
        print(f"\nPublic tunnel URL: {match.group(0)}")
        print("Copy this into ui/index.html's BASE_URL constant.\n")